# COMP813 Project — Underwater Plastic Bags vs Medusozoa
This notebook runs the project scripts on Google Colab.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive make a folder `COMP813` that contains:
   * `COMP813_Project_Code/` (the code folder)
   * `COMP813_data.zip` (jellyfish + synthetic bags)
   * `COMP813_trashcan.zip` (TrashCan 1.0, the real underwater bags)

The images are unzipped to Colab's fast local disk every session (reading thousands of small files straight from Drive is very slow). All results are saved to Drive, so nothing is lost if Colab disconnects.

## 0. Mount Drive, unzip the data, set the folders
If the whole group shares one Drive folder, everyone's results land in the same `outputs` folder.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/COMP813'
if not os.path.exists('/content/Task/archive_jelly'):
    !unzip -q -o "{PROJECT_DIR}/COMP813_data.zip" -d /content/Task
if not os.path.exists('/content/Task/archive_trashcan'):
    !unzip -q -o "{PROJECT_DIR}/COMP813_trashcan.zip" -d /content/Task

os.environ['DATA_ROOT'] = '/content/Task'             # raw images on the fast local disk
os.environ['OUT_DIR'] = f'{PROJECT_DIR}/outputs'      # results saved on Drive
%cd {PROJECT_DIR}/COMP813_Project_Code
!ls /content/Task

## 1. Install the libraries and check the GPU

In [ ]:
!pip -q install -r requirements.txt
import torch
print('GPU available:', torch.cuda.is_available())

## 2. Prepare the data
Cleans the data, cuts the bags out of the TrashCan video frames, splits 60/20/20 without leakage, builds the extra test sets and checks the class balance. Only needed once (about 1 minute).

In [ ]:
!python prepare_data.py
import json
s = json.load(open(os.environ['OUT_DIR'] + '/data_summary.json'))
print(json.dumps({k: s[k] for k in ['jellyfish_duplicates', 'trashcan_official_split', 'trashcan_bag_cleaning']}, indent=1))
for task in ['binary', 'fine']:
    for split in ['train', 'train_unique_photos_only', 'test']:
        b = s['class_balance'][task][split]
        print(task, split, b['counts'], 'imbalance ratio %.2f' % b['imbalance_ratio'])

In [ ]:
from IPython.display import Image, display
for f in ['data_class_balance', 'data_samples', 'data_trashcan_cleaning', 'data_underwater_severity', 'data_background_swap_examples']:
    display(Image(f"{os.environ['OUT_DIR']}/figures/{f}.png"))

## 3. Quick smoke test (few images, 1 epoch) — also good for the video demo
The numbers mean nothing; it only checks that every script runs (about 15 minutes).

In [ ]:
!python run_all.py --quick

## 4. Full experiments
Split these between the three members (each on their own Colab). Finished runs are skipped, so just run a cell again after a disconnect (run section 0 first).

| Member | Cells |
|---|---|
| A | 4a, 4b (seed 813), 4e |
| B | 4b with `--seeds 814`, 4c |
| C | 4b with `--seeds 815`, 4d |

In [ ]:
# 4a: classic baselines, CLIP and the learning-rate search (member A)
!python run_all.py --stages classic clip lrsweep

In [ ]:
# 4b: main models + underwater-domain augmentation (change the seed per member: 813 / 814 / 815)
!python run_all.py --stages main underwater --seeds 813

In [ ]:
# 4c: class-imbalance methods (all seeds)
!python run_all.py --stages imbalance --seeds 813 814 815

In [ ]:
# 4d: cross-dataset study (train on synthetic / both bag sources)
!python run_all.py --stages crossdata --seeds 813 814 815

In [ ]:
# 4e: ablations and data-size study
!python run_all.py --stages ablation datasize --seeds 813 814 815

## 5. Grad-CAM and summary (after all results are in `outputs/`)

In [ ]:
!python run_all.py --stages gradcam summary

In [ ]:
from IPython.display import Markdown
display(Markdown(open(f"{os.environ['OUT_DIR']}/summary/key_numbers.md").read()))

In [ ]:
import glob
out = os.environ['OUT_DIR']
for f in sorted(glob.glob(f'{out}/summary/*.png')) + sorted(glob.glob(f'{out}/figures/gradcam/*s813_background_swap.png'))[:3]:
    print(f); display(Image(f))